# Notebook13 source-bound workflow
Purpose: run registered fullJEPA geometryablation with fixedpairedseeds, allsixvariants andprospectivelyfixedprobe endpoints.
Inputs: taggedexperimentdefinitions, validationreceipts, unchangedconsumer. Outputs: artifacts/notebook_13/<run>, checkpoints/<run>. Up to12h orregistered3h; calibration timingalone selectsupdates.
Launch/resume: RunAll notebooks/13_Product_Geometry_JEPA.ipynb launch-product-geometry. Training tmux fmm-product-geometry-<run>; attach: tmux attach -t <session>, detach Ctrl-b d. Originalclock never resets. Definitionimports never launchtraining.


In [ ]:
"""Notebook13 orchestration: one persisted configured absolute experimental clock."""
from pathlib import Path
import copy, hashlib, importlib.util, json, os, shlex, subprocess, sys, time
from datetime import datetime, timezone
import nbformat
from fmm_jepa.bootstrap import TaggedNotebookLoader
from fmm_jepa.training import atomic_json, digest_file, run_writer
MAIN_NOTEBOOK='notebooks/13_Product_Geometry_JEPA.ipynb'
TERMINAL={'COMPLETED','FAILED','INCOMPLETE'}

def _read(path):return json.loads(Path(path).read_text())
def _normalized(value):return json.loads(json.dumps(value,allow_nan=False))
def _hash(value):return hashlib.sha256(json.dumps(_normalized(value),sort_keys=True,separators=(',',':')).encode()).hexdigest()
def _export_hash(path):
    return hashlib.sha256('\n'.join(''.join(c['source']) for c in _read(path)['cells'] if c['cell_type']=='code' and 'ahr-export' in c.get('metadata',{}).get('tags',[])).encode()).hexdigest()
def _load(root,name):
    key='fmm13_'+name
    if key not in sys.modules:
        spec=importlib.util.spec_from_loader(key,TaggedNotebookLoader(Path(root)/'experiments/13_product_geometry'/(name+'.ipynb')))
        module=importlib.util.module_from_spec(spec);sys.modules[key]=module
        try:spec.loader.exec_module(module)
        except BaseException:sys.modules.pop(key,None);raise
    return sys.modules[key]

def source_hashes(root):
    folders=('fmm_jepa','experiments/04_followup','experiments/05_scaling','experiments/07_energy','experiments/08_replication','experiments/09_deep_energy','experiments/10_manifold_kernel','experiments/11_manifold_long','experiments/13_product_geometry')
    return {str(p.relative_to(root)):_export_hash(p) for folder in folders for p in sorted((Path(root)/folder).glob('*.ipynb'))}

def consumer_hash(root):return _hash([''.join(c['source']) for c in _read(Path(root)/MAIN_NOTEBOOK)['cells'] if c['cell_type']=='code'])

def study_configuration(run_id='geometry-ablation-001',budget_seconds=43200):
    if not run_id or any(not(c.isalnum() or c in '-_') for c in run_id):raise ValueError('Safe nonempty run ID required')
    budget_seconds=int(budget_seconds)
    if budget_seconds not in (43200,10800):raise ValueError('Only prospectively selected12h or3h configurations supported')
    full=budget_seconds==43200
    return {'schema':'fmm-jepa-notebook13-study/v1','run_id':run_id,'hard_limit_seconds':budget_seconds,
        'device':'cuda:0','dtype':'float32','image_size':128,'effective_batch':128,'microbatch':128,
        'seeds':list(range(20261501,20261509 if full else 20261507)),
        'variants':['vit','learned_kernel','normalized_kernel','product_kernel','sphere_kernel','multi_view_kernel'],
        'update_candidates':[1000,800,600] if full else [600,350,200],
        'probe_endpoints':['initial','final'] if full else ['final'],
        'round_updates':50,'probe_epochs':20,'heldout_enabled':False,
        'input_pipeline':{'workers':6},'execution_optimization':True,
        'kernel_geometry':{'depth':4,'heads':3,'coordinate_dim':8,'semantic_dim':6,'views':3,'bandwidths':[.15,.35,.75],'content_scale':.5,'checkpoint':False},
        'analysis':{'role':'registered_primary_Holm_family_with_exploratory_secondary','n_pairs':8 if full else 6,
            'df':7 if full else 5,'confidence_level':.95,'family_alpha':.05,'multiplicity':'Holm',
            'contrasts':['product_kernel_minus_vit','sphere_kernel_minus_vit','multi_view_kernel_minus_vit'],
            'endpoint':'Fixed epoch20 frozen-EMA-probe development top1 at final common successful-update budget, percentage points',
            'interpretation':'Training seeds are paired independent replication units; images reused. No guaranteed significance, novelty, intrinsicmanifold identification or universal advantage.'},
        'primary_analysis':{'role':'registered_primary_family','n_pairs':8 if full else 6,'df':7 if full else 5,'alpha':.05,'multiplicity':'Holm',
            'family':['product_kernel_minus_vit_final','sphere_kernel_minus_vit_final','multi_view_kernel_minus_vit_final'],
            'tests':['paired_t_two_sided_Holm_three','exact_two_sided_sign_flip_secondary','exact_two_sided_sign_secondary'],
            'endpoint':'Final fixedepoch20 probe; updates selected from timing alone before quality outcomes.'},
        'secondary_analysis':'Originallearned/normalized controls, geometry-versuscontrol contrasts, owninitial gains only12hdesign, representation/geometry diagnostics and measured cost are exploratory.',
        'budget_policy':'Allseedpairs/all6architectures fixed. Highest predeclared updatecandidate fitting originalbudget after timing-only calibration,10percent trainingmargin, per50update checkpoint/reconstruction overhead, historicalevaluationreserve andpreflight/recovery reserve. Never extend clock or remove failingarms.',
        'selection':'Fullsixvariant fixedepoch20 architecturecomparison. Originalselected2020development foriteration; opened8074reserve isnotfresh. Official25246test remainsclosed.',
        'interpretation':'Full customJEPA context/EMA encoders changed; sameTransformerpredictor/masks/loss/EMA/optimizer. Exactdense sums, noFMMspeedup orimage-featureisomorphism claim.',
        'heartbeat':{'interval_seconds':3600,'poll_seconds':30,'stall_seconds':1200,'automatic_resume_attempts':3,'agent_review_enabled':True,'agent_timeout_seconds':900},
        'visual_review':'notebooks/13_Results_Review.ipynb'}



def _status(artifact,stage,status,**details):
    result={'stage':stage,'status':status,'updated_unix':time.time(),**details}
    atomic_json(Path(artifact)/'study_status.json',result)
    print(stage+': '+status,flush=True)
    return result

def _validation(root):
    root=Path(root);result={}
    rows=[('operator','13_product_geometry','notebook_13',['encoder']),
          ('input_pipeline','11_manifold_long','notebook_11',['input_pipeline']),
          ('runner_evaluation','13_product_geometry','notebook_13',['runner','evaluation']),
          ('execution_optimization','11_manifold_long','notebook_11',['engineering']),
          ('heartbeat','13_product_geometry','notebook_13',['heartbeat']),
          ('visual_report','13_product_geometry','notebook_13',['visualization'])]
    for stem,folder,artifact,files in rows:
        p=root/'artifacts'/(artifact+'_validation')/(stem+'_validation.json');report=_read(p)
        if report.get('status')!='PASS':raise ValueError('Notebook13 requires PASS '+stem+' validation')
        for name in files:
            if report.get(name+'_export_sha256')!=_export_hash(root/'experiments'/folder/(name+'.ipynb')):raise ValueError('Validation source differs: '+name)
        result[stem]={'path':str(p.relative_to(root)),'sha256':digest_file(p)}
    agent=root/'artifacts/notebook_13_validation/agent_probe/probe_result.json';report=_read(agent)
    if report.get('status')!='PASS':raise ValueError('Hourly agent authentication/tool probe must pass')
    result['hourly_agent']={'path':str(agent.relative_to(root)),'sha256':digest_file(agent)}
    return result



def _check_binding(root,binding,config):
    reference='artifacts/notebook_07/energy-encoder-001/stages/learning/plan.json'
    if binding.get('reference_pilot_plan')!={'path':reference,'sha256':digest_file(Path(root)/reference)}:
        raise ValueError('The pilot configuration source changed')
    expected_timing={str(p.relative_to(root)):digest_file(p) for p in (Path(root)/'artifacts/notebook_11/manifold-long-001/stages/learning/stage_evaluation.json',Path(root)/'artifacts/notebook_11/manifold-long-001/stages/learning/calibration_result.json')}
    if binding.get('timing_reference_files')!=expected_timing:raise ValueError('Registered historicaltiming reference changed')
    if binding['configuration']!=_normalized(config):raise ValueError('Registered run configuration changed')
    if binding['source_hashes']!=source_hashes(root):raise ValueError('Registered source definitions changed; use a new run ID')
    if binding['consumer_notebook_sha256']!=consumer_hash(root):raise ValueError('Registered main notebook code changed')
    if binding['operator_validation']!=_validation(root):raise ValueError('Bound validation evidence changed')

def _clock(artifact,binding):
    result=_read(Path(artifact)/'budget_clock.json')
    if result!=binding['budget_clock'] or result['max_seconds']!=binding['configuration']['hard_limit_seconds']:raise ValueError('Original registered clock changed')
    return result

def _prepare(root,config):
    if _normalized(config)!=_normalized(study_configuration(config['run_id'],config['hard_limit_seconds'])):
        raise ValueError('This pilot has a fixed registered configuration; no seed/update/architecture edits are permitted')
    root=Path(root).resolve();artifact=root/'artifacts/notebook_13'/config['run_id'];artifact.mkdir(parents=True,exist_ok=True)
    path=artifact/'study_binding.json'
    if path.exists():
        binding=_read(path);_check_binding(root,binding,config)
    else:
        if (artifact/'budget_clock.json').exists() or (artifact/'stages').exists():raise ValueError('Unbound existing experiment evidence; refusing restart')
        validation=_validation(root);now=time.time()
        clock={'started_unix':now,'deadline_unix':now+config['hard_limit_seconds'],'max_seconds':config['hard_limit_seconds'],'policy':'Original launch clock includes calibration, recovery verification, training and evaluation; never resets.'}
        binding={'schema':'notebook13-binding/v1','configuration':_normalized(config),'source_hashes':source_hashes(root),
                 'consumer_notebook_sha256':consumer_hash(root),'budget_clock':clock,'operator_validation':validation,
                 'timing_reference_files':{str(p.relative_to(root)):digest_file(p) for p in (root/'artifacts/notebook_11/manifold-long-001/stages/learning/stage_evaluation.json',root/'artifacts/notebook_11/manifold-long-001/stages/learning/calibration_result.json')},
                 'reference_pilot_plan':{'path':'artifacts/notebook_07/energy-encoder-001/stages/learning/plan.json',
                    'sha256':digest_file(root/'artifacts/notebook_07/energy-encoder-001/stages/learning/plan.json')}}
        atomic_json(path,binding);atomic_json(artifact/'budget_clock.json',clock)
    return artifact,binding,_clock(artifact,binding)

def _choose_budget(config,calibration,deadline,root=None):
    if calibration['selected_microbatch']!=128 or calibration['selected_accumulation_steps']!=1:raise ValueError('Physical/effectivebatch128 fixed')
    timings=calibration['selected_model_step_seconds'];overhead=calibration['selected_model_round_overhead_seconds']
    if set(timings)!=set(config['variants']) or set(overhead)!=set(config['variants']):raise ValueError('Allsixvariants require timing andoverhead calibration')
    if any(not isinstance(t,(int,float)) or not(0<t<float('inf')) for t in timings.values()):raise ValueError('Invalid step timings')
    if any(not isinstance(t,(int,float)) or not(0<=t<float('inf')) for t in overhead.values()):raise ValueError('Invalid round overhead')
    count=len(config['seeds']);probes=count*len(config['variants'])*len(config['probe_endpoints']);remaining=deadline-time.time()
    source_root=Path(root) if root is not None else Path(__file__).resolve().parents[2]
    history_path=source_root/'artifacts/notebook_11/manifold-long-001/stages/learning/stage_evaluation.json'
    history=_read(history_path);per_probe=(history['completed_unix']-history['started_unix'])/len(history['evaluations'])
    if not 0<per_probe<float('inf'):raise ValueError('Historical probe timing invalid')
    # New geometry may cost more than old learned; conservatively scale fullprobe timing.
    old_steps=_read(source_root/'artifacts/notebook_11/manifold-long-001/stages/learning/calibration_result.json')['selected_model_step_seconds']
    ratio=max(1.,max(timings.values())/max(old_steps.values()))
    reserve=max(1200.,1.5*probes*per_probe*ratio)+300.
    preflight_recovery=1800 if config['hard_limit_seconds']==43200 else 600
    choices=[]
    for updates in config['update_candidates']:
        rounds=(updates+config['round_updates']-1)//config['round_updates']
        training=count*(updates*sum(timings.values())+rounds*sum(overhead.values()))
        predicted=1.10*training
        choices.append({'updates':updates,'predicted_training_seconds':predicted,'predicted_total_remaining_seconds':predicted+reserve+preflight_recovery,'rounds':rounds})
    selected=next((x for x in choices if x['predicted_total_remaining_seconds']<=remaining),None)
    if selected is None:raise TimeoutError('No fullregistered sixvariant pairedstudy fits originalcap; noquality-based smallerstudy: '+str(choices))
    return {'status':'PASS',**selected,'seeds':count,'architectures':len(config['variants']),'planned_probes':probes,
        'evaluation_reserve_seconds':reserve,'preflight_recovery_reserve_seconds':preflight_recovery,'remaining_seconds':remaining,
        'calibration_round_seconds':sum(timings.values()),'calibration_round_overhead_seconds':sum(overhead.values()),
        'historical_evaluation_timing':{'path':str(history_path.relative_to(source_root)),'sha256':digest_file(history_path),'seconds_per_probe':per_probe,'new_geometry_step_cost_multiplier':ratio},
        'timing_only_candidates':choices,'selection_rule':'Largest predeclared candidate fitting originalabsolute clock; quality evidence forbidden.'}


def _plan(root,study,binding,bundle,base,calibration,budget):
    preparation=_load(root,'preparation');arms=[];seeds=study['seeds'];updates=budget['updates']
    for index,seed in enumerate(seeds):
        for variant in study['variants']:
            cfg=preparation.arm_config(base,variant)
            cfg['kernel13'].update(copy.deepcopy(study['kernel_geometry']));cfg['kernel13']['variant']=variant
            cfg['training'].update(updates=updates,microbatch=calibration['selected_microbatch'],accumulation_steps=calibration['selected_accumulation_steps'])
            cfg['seeds'].update(model_init=seed,data_order=971101+1000*index,augmentation=972202+1000*index,mask=973303+1000*index,probe=4404)
            cfg['model']['feature_seed']=seed+1000
            cfg['evaluation'].update(batch_size=min(64,calibration['selected_microbatch']),diagnostic_steps=[0,updates])
            cfg['budget']['evaluation_reserve_seconds']=budget['evaluation_reserve_seconds']
            cfg['input_pipeline']=copy.deepcopy(study['input_pipeline'])
            cfg['execution_optimization']=study['execution_optimization']
            arms.append({'id':'s'+str(seed)+'-'+variant,'kind':'vit' if variant=='vit' else 'semantic','variant':variant,'seed':seed,'condition':variant,'config':cfg})
    return _normalized({'schema':'notebook13-stage-plan/v1','run_id':study['run_id'],'stage_id':'learning','stage_role':'learning',
         'stage':{'id':'learning','image_size':128,'pretrain_pool':'selected'},'updates':updates,'milestones':[updates],'probe_endpoints':study['probe_endpoints'],'hard_limit_seconds':study['hard_limit_seconds'],
         'round_updates':study['round_updates'],'effective_batch':128,'seeds':seeds,'variants':study['variants'],'arms':arms,
         'train_count':len(bundle['train_ids']),'probe_train_count':len(bundle['probe_train_ids']),
         'source_hashes':binding['source_hashes'],'consumer_notebook_sha256':binding['consumer_notebook_sha256'],
         'resume_notebook':MAIN_NOTEBOOK,'resume_cell':'launch-product-geometry','evaluation_reserve_seconds':budget['evaluation_reserve_seconds'],
         'study_configuration_hash':_hash(study),'numerical_backend':'exact-dense-product-geometry-v1','heldout_status':'NOT_RUN',
         'analysis':study['analysis'],'primary_analysis':study['primary_analysis'],'secondary_analysis':study['secondary_analysis'],
         'excluded_pilot_seeds':[20261011,20261012,20261201,20261202,20261301,20261302]+list(range(20261101,20261111))+list(range(20261401,20261407)),
         'selection':study['selection'],'capacity_policy':'Shared stem/allfourMLPs/norms/predictor; allkernel controls share value/projection/bandwidth/mixture andcomparablebasegeometry initialization; addedparameters active andreported.',
         'failure_policy':'Stop first numerical failure; never skip samples.','budget_selection':budget})

def run_study(root,run_id):
    root=Path(root).resolve();artifact=root/'artifacts/notebook_13'/run_id
    binding=_read(artifact/'study_binding.json');study=binding['configuration'];_check_binding(root,binding,study)
    clock=_clock(artifact,binding);deadline=clock['deadline_unix']
    if (artifact/'study_status.json').exists() and _read(artifact/'study_status.json').get('status')=='COMPLETED':return _read(artifact/'study_status.json')
    if time.time()>=deadline:raise TimeoutError('Original registered deadline expired')
    preparation,runner,evaluation=(_load(root,n) for n in ('preparation','runner','evaluation'))
    with run_writer(artifact):
        try:
            folder=artifact/'stages/learning';folder.mkdir(parents=True,exist_ok=True)
            (folder/'predictions').mkdir(exist_ok=True)
            base=preparation.base_config(root)
            base['kernel13'].update(copy.deepcopy(study['kernel_geometry']))
            base['input_pipeline']=copy.deepcopy(study['input_pipeline'])
            base['execution_optimization']=study['execution_optimization']
            base['budget'].update(max_seconds=study['hard_limit_seconds'],target_seconds=study['hard_limit_seconds'])
            _status(artifact,'verify_training_and_development_data','RUNNING')
            bundle=preparation.prepare_bundle(root,folder,base)
            _status(artifact,'actual_update_calibration','RUNNING')
            calibration=preparation.calibrate(root,folder,base,bundle,deadline)
            plan_path=folder/'plan.json'
            if plan_path.exists():
                plan=_read(plan_path)
                if plan['source_hashes']!=binding['source_hashes']:raise ValueError('Saved plan sources changed')
                if plan['study_configuration_hash']!=_hash(study):raise ValueError('Saved plan study changed')
            else:
                budget=_choose_budget(study,calibration,deadline,root)
                plan=_plan(root,study,binding,bundle,base,calibration,budget)
                preparation.immutable(plan_path,plan)
            _load(root,'visualization').register_analysis13(root,folder,plan)
            _status(artifact,'exact_resume_preflight','RUNNING',updates=plan['updates'],seeds=plan['seeds'],variants=plan['variants'])
            runner.prepare_stage(root,folder,plan,bundle)
            preflight=runner.verify_stage_resume(root,folder,plan,bundle,deadline)
            atomic_json(folder/'preflight_result.json',preflight)
            if preflight.get('status')!='PASS':raise RuntimeError('Exact-resume preflight did not pass')
            _status(artifact,'evaluation_integration_preflight','RUNNING')
            scoring=evaluation.verify_evaluation_smoke(root,folder,plan,bundle,deadline)
            atomic_json(folder/'evaluation_preflight_result.json',scoring)
            if scoring.get('status')!='PASS':raise RuntimeError('Evaluation integration preflight failed')
            _check_binding(root,binding,study)
            _status(artifact,'paired_training','RUNNING',updates=plan['updates'],seeds=plan['seeds'],variants=plan['variants'])
            training_path=folder/'training_result.json'
            if training_path.exists() and _read(training_path).get('status')=='COMPLETE':
                # Preserve the authoritative training record and evaluation identity on evaluation-only recovery.
                training=_read(training_path)
                evaluation._stage_jobs(plan,training)
                from fmm_jepa.food101_training import validate_training_manifest
                for arm in plan['arms']:
                    directory=root/'checkpoints'/plan['run_id']/'stages/learning/arms'/arm['id']
                    checkpoint=validate_training_manifest(directory)
                    if checkpoint['successful_updates']!=plan['updates'] or checkpoint['attempted_updates']!=plan['updates']:
                        raise ValueError('Completed training record/checkpoint differs')
                    runner.validate_stage_milestones(directory,plan,required=plan['milestones'])
            else:
                training=runner.train_stage(root,folder,plan,bundle,deadline)
                atomic_json(training_path,training)
            _status(artifact,'frozen_feature_evaluation','RUNNING',updates=plan['updates'],seeds=plan['seeds'])
            result=evaluation.evaluate_stage(root,folder,plan,bundle,training,deadline)
            visual=_load(root,'visualization').build_visual_report(root,folder,plan,bundle,result,deadline_unix=deadline)
            result['visual_review']=visual
            atomic_json(artifact/'study_results.json',result)
            if result.get('status') not in ('PASS','COMPLETE','COMPLETED'):raise TimeoutError('Evaluation incomplete under original deadline')
            _check_binding(root,binding,study)
            atomic_json(folder/'stage_evaluation.json',result)
            return _status(artifact,'complete','COMPLETED',elapsed_seconds=time.time()-clock['started_unix'],updates=plan['updates'],seeds=plan['seeds'],variants=plan['variants'],heldout_status='NOT_RUN',results=str(folder/'stage_results.md'))
        except BaseException as error:
            _status(artifact,'interrupted','INCOMPLETE' if isinstance(error,TimeoutError) else 'FAILED',error=type(error).__name__+': '+str(error),cause=repr(error.__cause__) if error.__cause__ else None,elapsed_seconds=time.time()-clock['started_unix'])
            raise

def _exists(session):return subprocess.run(['tmux','has-session','-t','='+session],capture_output=True).returncode==0

def launch_study(root,config):
    root=Path(root).resolve();artifact=root/'artifacts/notebook_13'/config['run_id']
    with run_writer(artifact/'launch_lock'):
        artifact,binding,clock=_prepare(root,config);session='fmm-product-geometry-'+config['run_id']
        if _exists(session):
            _load(root,'deadline_watchdog').launch_watchdog(root,config['run_id'])
            _load(root,'heartbeat').launch_heartbeat(root,config['run_id'])
            return {'status':'EXISTING_SESSION','session':session,'deadline_unix':clock['deadline_unix'],'artifact':str(artifact)}
        prior=_read(artifact/'study_status.json') if (artifact/'study_status.json').exists() else {}
        if prior.get('status')=='COMPLETED':return prior
        if time.time()>=clock['deadline_unix']:raise TimeoutError('Original deadline expired; Run All cannot reset it')
        setup=next(c.source for c in nbformat.read(root/MAIN_NOTEBOOK,as_version=4).cells if c.get('id')=='bootstrap-product-geometry' and 'ahr-export' in c.metadata.get('tags',[]))
        code=setup+'\nWORKFLOW.run_study(ROOT, '+repr(config['run_id'])+')\n'
        worker=artifact/'worker.ipynb'
        if worker.exists():
            if [c.source for c in nbformat.read(worker,as_version=4).cells if c.cell_type=='code']!=[code]:raise ValueError('Detached worker source changed')
        else:
            nbformat.write(nbformat.v4.new_notebook(cells=[nbformat.v4.new_markdown_cell('# Notebook13 detached worker\nOutputs: this run folder; full-state checkpoints: checkpoints/'+config['run_id']+'/. Resume with unchanged Notebook13 Run All before the original registered deadline.'),nbformat.v4.new_code_cell(code)],metadata={'kernelspec':{'display_name':'Python (ahr-go)','language':'python','name':'ahr-go'}}),worker)
        command=[sys.executable,'-m','jupyter','nbconvert','--execute','--to','notebook','--inplace','--ExecutePreprocessor.timeout=-1','--ExecutePreprocessor.kernel_name=ahr-go',str(worker)]
        shell='exec '+shlex.join(command)+' > '+shlex.quote(str(artifact/'worker.log'))+' 2>&1'
        launch={'session':session,'artifact':str(artifact),'command':command,'attach':'tmux attach -t '+session,'detach':'Ctrl-b, then d','deadline_unix':clock['deadline_unix'],'deadline_utc':datetime.fromtimestamp(clock['deadline_unix'],timezone.utc).isoformat(),'binding_hash':_hash(binding)}
        atomic_json(artifact/'launch.json',launch);_status(artifact,'launching','STARTING')
        try:
            _load(root,'deadline_watchdog').launch_watchdog(root,config['run_id'])
            _load(root,'heartbeat').launch_heartbeat(root,config['run_id'])
            subprocess.run(['tmux','new-session','-d','-s',session,'-c',str(root),'bash','-c',shell],check=True)
        except BaseException as error:_status(artifact,'launch_failed','FAILED',error=repr(error));raise
        return {'status':'LAUNCHED',**launch}

def inspect_study(root,run_id):
    artifact=Path(root)/'artifacts/notebook_13'/run_id
    if not (artifact/'study_status.json').exists():return {'status':'NOT_LAUNCHED'}
    status=_read(artifact/'study_status.json');clock=_read(artifact/'budget_clock.json')
    result={**status,'session':'fmm-product-geometry-'+run_id,'deadline_utc':datetime.fromtimestamp(clock['deadline_unix'],timezone.utc).isoformat(),'remaining_minutes':max(0,(clock['deadline_unix']-time.time())/60)}
    rows=[]
    for p in sorted((Path(root)/'checkpoints'/run_id/'stages/learning').glob('arms/*/manifest.json')):
        m=_read(p);rows.append({'arm':p.parent.name,'successful_updates':m.get('successful_updates',m.get('latest_step'))})
    result['saved_training_progress']=rows
    return result


def readiness(root,config):
    """Read-only prelaunch check; creates no run folder, budget clock or tmux worker."""
    root=Path(root).resolve()
    if _normalized(config)!=_normalized(study_configuration(config['run_id'],config['hard_limit_seconds'])):raise ValueError('Registered configuration changed')
    validation=_validation(root)
    encoder=_load(root,'encoder');preparation=_load(root,'preparation')
    base=preparation.base_config(root);rows=[]
    from fmm_jepa.food101_training import isolated_rng
    for variant in config['variants']:
        cfg=copy.deepcopy(base['model']);cfg['kernel13']={**config['kernel_geometry'],'variant':variant}
        with isolated_rng(base['seeds']['model_init']):model=encoder.build_model(cfg,kind='vit' if variant=='vit' else 'semantic')
        counts=encoder.parameter_counts(model)
        total=counts['encoder_total'];inactive=counts['encoder_inactive_scaffold']
        rows.append({'architecture':variant,'depth':len(model.context.blocks),'encoder_parameters_total':total,
            'encoder_parameters_active':total-inactive,'inactive_parameters':inactive,
            'trainable_parameters_total':sum(p.numel() for p in model.parameters() if p.requires_grad)})
        del model
    artifact=root/'artifacts/notebook_13'/config['run_id']
    started=(artifact/'budget_clock.json').exists()
    current=_read(artifact/'study_status.json') if (artifact/'study_status.json').exists() else {}
    scientific='COMPLETE' if current.get('status')=='COMPLETED' else ('NOT_RUN' if not started else current.get('status','IN_PROGRESS'))
    return {'status':'EXISTING_STUDY' if started else 'READY_TO_LAUNCH','production_run_started':started,'expected_runtime':str(config['hard_limit_seconds']/3600)+'hours maximum includingpreflight,training,allregisteredprobes,visualreport andrecovery; timing-selectedcomplete schedule.',
        'architecture_summary':rows,'validation':validation,'run_id':config['run_id'],'planned_training_runs':len(config['seeds'])*len(config['variants']),'planned_probe_evaluations':len(config['seeds'])*len(config['variants'])*len(config['probe_endpoints']),
        'checkpoint_location':str(root/'checkpoints'/config['run_id']), 'scientific_results':scientific}
